# Solar sail: first-order results

Compares the saved first-order runs in `results/`. Nothing is solved here: each run is loaded from its
`solution.npz`, so re-run this notebook after a new run finishes. Runs that don't exist yet are skipped.

To reproduce a run from this folder:

```
python main.py algorithms/ms_term_fo.yaml --no-search -n 21 --output results/ms_term_fo
```

**Reading the results**
- `autoscvx` buffers the dynamics. `term` is `autoscvx-term`, with hard linearized dynamics and buffered terminal conditions.
- A flow saves only its final stage, so its iteration count and history cover that stage alone. Per-stage counts are printed at the end of each run.
- The "true" columns come from independent re-propagation. The plots show node values.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

from solution import load_solution, validation_report

RESULTS = Path("results")

# run folder -> (label, color). Colors stay with the run, so a missing run never repaints the others.
RUNS = {
    "ms_fo":            ("MS, autoscvx",  "#2a78d6"),
    "p_fo":             ("p, autoscvx",   "#eb6834"),
    "ms_term_fo":       ("MS, term",      "#1baf7a"),
    "ms_to_hp_term_fo": ("MS → hp, term", "#eda100"),
    "p_term_w100_fo":   ("p, term",       "#e87ba4"),
    "p_to_ms_term_fo":  ("p → MS, term",  "#008300"),
}

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 10, "lines.linewidth": 2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
})

## Load the runs

In [ ]:
runs = {}
for name, (label, color) in RUNS.items():
    path = RESULTS / name / "solution.npz"
    if not path.exists():
        print(f"not found yet: {path}")
        continue
    solution = load_solution(path)
    runs[name] = dict(label=label, color=color, solution=solution, report=validation_report(solution))
print(f"loaded {len(runs)} of {len(RUNS)} runs")

## Summary

Requirements: true perigee at least 250 km above GEO, true eccentricity at most 0.003, accumulated defect at most 3 km.
A run is validated only if every check in `validation_report` passes.

In [ ]:
rows = ["| Run | Validated | Solver converged | Iterations (last stage) | Transfer (d) | True perigee above GEO (km) | True e | Defect (km) | Max re-prop error |",
        "|---|---|---|---:|---:|---:|---:|---:|---:|"]
for run in runs.values():
    s, r = run["solution"], run["report"]
    rows.append(
        f"| {run['label']} | {'yes' if r['numerically_feasible'] else 'no'} | {'yes' if r['solver_converged'] else 'no'} "
        f"| {r['iterations']} | {r['elapsed_days']:.4f} | {r['true_final_perigee_km'] - s.geo_radius_km:+.2f} "
        f"| {r['true_final_eccentricity']:.6f} | {r['accumulated_defect_km']:+.3f} | {r['max_propagation_error_nd']:.2e} |")
display(Markdown("\n".join(rows)))

## Orbit raising (node values)

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for run in runs.values():
    s = run["solution"]
    left.plot(s.elapsed_days, s.perigee_radius_km - s.geo_radius_km, color=run["color"], label=run["label"])
    right.plot(s.elapsed_days, s.eccentricity, color=run["color"], label=run["label"])

any_run = next(iter(runs.values()))["solution"]
left.axhline(any_run.target_perigee_km - any_run.geo_radius_km, color="#6b6a65", linestyle="--", linewidth=1, label="Requirement")
right.axhline(any_run.target_eccentricity, color="#6b6a65", linestyle="--", linewidth=1, label="Requirement")
left.set(title="Perigee above GEO", xlabel="Elapsed time (days)", ylabel="km")
right.set(title="Eccentricity", xlabel="Elapsed time (days)", ylabel="e")
left.legend(frameon=False)
fig.tight_layout()
plt.show()

## Convergence

History of the final stage of each run. A flat curve that stays above its starting trend means the iteration has stopped improving.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for run in runs.values():
    s = run["solution"]
    for axis, history in ((left, s.defect_history), (right, s.step_history)):
        history = np.asarray(history, dtype=float)
        axis.semilogy(np.arange(1, history.size + 1), history, color=run["color"], label=run["label"])
left.set(title="Dynamics defect", xlabel="Iteration", ylabel="Non-dimensional")
right.set(title="Step size", xlabel="Iteration", ylabel="Non-dimensional")
left.legend(frameon=False)
fig.tight_layout()
plt.show()

## Saved figures for one run

Set `RUN` to any folder name from the table of runs above. These are the figures `main.py` wrote for that run.

In [ ]:
RUN = "ms_term_fo"

for figure in ("summary", "transfer_plane", "elements", "control", "phases", "convergence"):
    path = RESULTS / RUN / f"{figure}.png"
    if path.exists():
        display(Markdown(f"### {RUNS[RUN][0]}: {figure}"))
        display(Image(filename=path, width=900))